In [1]:
import subprocess, sys

BITSANDBYTES_PIN = "0.46.1"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"bitsandbytes=={BITSANDBYTES_PIN}",
)
print("profiling pins installed (no vLLM today)")


installing: transformers==4.46.* accelerate==1.1.* bitsandbytes==0.46.1
profiling pins installed (no vLLM today)


In [2]:
import torch, gc
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

def resident_vram_gb() -> float:
    torch.cuda.synchronize()
    return round(torch.cuda.memory_reserved() / (1024 ** 3), 2)

def load(dtype: str):
    if dtype == "fp16":
        return AutoModelForCausalLM.from_pretrained(
            MODEL, torch_dtype=torch.float16, device_map="cuda"
        )
    if dtype == "int8":
        qc = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=qc, device_map="cuda"
        )
    if dtype == "int4":
        qc = BitsAndBytesConfig(load_in_4bit=True)
        return AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=qc, device_map="cuda"
        )
    raise ValueError(f"Unknown dtype: {dtype}")

In [3]:
def unload(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

for dtype in ["fp16", "int8", "int4"]:
    model = load(dtype)
    vram = resident_vram_gb()
    print(dtype, vram)
    unload(model)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

fp16 3.06
int8 4.79
int4 2.88


In [4]:
results = {}

for dtype in ["fp16", "int8", "int4"]:
    model = load(dtype)
    vram = resident_vram_gb()
    results[dtype] = vram
    print(dtype, vram)
    del model
    gc.collect()
    torch.cuda.empty_cache()

fp16_gb = results["fp16"]
int8_gb = results["int8"]
int4_gb = results["int4"]


fp16 4.22
int8 1.74
int4 1.15


In [5]:
assert int8_gb < fp16_gb, "int8 should be smaller than fp16"
assert int4_gb < int8_gb, "int4 should be smaller than int8"
print("GREEN CHECK: PASS")

GREEN CHECK: PASS
